<a href="https://colab.research.google.com/github/nooyeat/kaggle/blob/main/%EC%B1%84%EB%AC%B4%20%EB%B6%88%EC%9D%B4%ED%96%89%20%EB%B6%84%EB%A5%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import pandas as pd
from imblearn.over_sampling import SMOTE
df = pd.read_csv('train.csv')
df = df.drop('UID', axis=1)
test = pd.read_csv('test.csv')
test = test.drop('UID', axis=1)
df.head()

,주거 형태,연간 소득,현재 직장 근속 연수,체납 세금 압류 횟수,개설된 신용계좌 수,신용 거래 연수,최대 신용한도,신용 문제 발생 횟수,마지막 연체 이후 경과 개월 수,개인 파산 횟수,대출 목적,대출 상환 기간,현재 대출 잔액,현재 미상환 신용액,월 상환 부채액,신용 점수,채무 불이행 여부
0,자가,1941337.5,10년 이상,0.0,9,13.4,400597.5,0,24,1,부채 통합,단기 상환,390903.0,225457.5,8806.5,767,0
1,월세,1979505.0,10년 이상,0.0,5,15.1,360679.5,0,11,0,부채 통합,단기 상환,1002184.5,64749.0,24961.5,767,0
2,월세,1356381.0,4년,0.0,12,18.8,491770.5,1,74,3,부채 통합,단기 상환,227775.0,487644.0,12069.0,800,1
3,월세,1049017.5,6년,0.0,15,14.8,411546.0,1,22,1,부채 통합,단기 상환,251383.5,413211.0,31749.0,796,1
4,월세,4320217.5,2년,0.0,11,26.1,895288.5,0,32,0,부채 통합,장기 상환,1163176.5,78991.5,5862.0,751,0


In [ ]:
# 현재 직장 근속 연수 -> 숫자만 추출
def year_int(x):
    return re.findall(r'\d+', x)[0]

from sklearn.preprocessing import LabelEncoder, MinMaxScaler

le = LabelEncoder()

df['현재 직장 근속 연수'] = df['현재 직장 근속 연수'].apply(year_int)
df = pd.get_dummies(df, columns=['주거 형태', '대출 목적', '대출 상환 기간'])

test['현재 직장 근속 연수'] = test['현재 직장 근속 연수'].apply(year_int)
test = pd.get_dummies(test, columns=['주거 형태', '대출 목적', '대출 상환 기간'])

In [ ]:
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import roc_auc_score
import numpy as np

X = df.drop('채무 불이행 여부', axis=1)
y = df['채무 불이행 여부']


log_columns = ["현재 미상환 신용액", "월 상환 부채액", "현재 대출 잔액"]
for col in log_columns:
    X[col] = np.log1p(X[col])
    test[col] = np.log1p(test[col])

mm = MinMaxScaler()
X = mm.fit_transform(X)

mm = MinMaxScaler()
test_scaled = mm.fit_transform(test)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)

model = keras.Sequential([
    layers.Dense(128, activation='leaky_relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='leaky_relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='leaky_relu'),
    layers.Dense(1, activation='sigmoid')
])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta=0.001,
    restore_best_weights=True
)

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0015),
              loss='binary_crossentropy', metrics=['AUC'])

history = model.fit(X_train, y_train, epochs=100, validation_data=(X_test, y_test),
                    batch_size=32, callbacks=[early_stopping], verbose=1)

Epoch 1/100
289/289 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - AUC: 0.6169 - loss: 0.6711 - val_AUC: 0.7090 - val_loss: 0.6246
Epoch 2/100
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.6919 - loss: 0.6360 - val_AUC: 0.7207 - val_loss: 0.6183
Epoch 3/100
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.7010 - loss: 0.6290 - val_AUC: 0.7216 - val_loss: 0.6188
Epoch 4/100
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.7109 - loss: 0.6216 - val_AUC: 0.7241 - val_loss: 0.6192
Epoch 5/100
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.7166 - loss: 0.6161 - val_AUC: 0.7299 - val_loss: 0.6087
Epoch 6/100
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.7230 - loss: 0.6115 - val_AUC: 0.7286 - val_loss: 0.6091
Epoch 7/100
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.7172 - loss: 0.6160 - val_AUC: 0.7344 - val_loss: 0.6095
Epoch 8/100
289/289 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - AUC: 0.7234 - loss: 0.6105 - val_AUC: 0.7356 - val_loss: 0.6106
Epoch 9/100
289/289 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms

In [ ]:
test_pred_prob = model.predict(test_scaled)
test_pred = (test_pred_prob >= 0.5).astype(int)

sub = pd.read_csv('sample_submission.csv')
sub['채무 불이행 확률'] = test_pred

sub.to_csv('submission.csv', index=False)

65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
a = pd.read_csv('submission.csv')
a['채무 불이행 확률'].value_counts()

,count
채무 불이행 확률,
0,1507
1,555
